# GenAI + RAG: explaining the maps in plain language

Member 4, step 2 (slide 22). `urbanheat.app.assistant`:
- **Facts** come from the model and the XAI for the place the user selected (predicted vs observed anomaly, top
  factors in °C, scenario result). The language model never makes up numbers.
- **Retrieval (RAG):** the project's own documents (`docs/KNOWLEDGE.md`, `docs/CONTRACTS.md`,
  `docs/HANDOVER_MEMBER3.md`) are split into one passage per section and indexed with TF-IDF (small, domain-specific
  corpus, no embedding service needed). The best 4 passages go into the prompt and are cited as [1]–[4].
- **Generation:** Claude (`claude-opus-5-5`, low effort, server-side fallback if a request is declined) writes a
  grounded answer of under 200 words, with the caveats (surface, not air temperature; model estimates, not causes).
- **Offline:** without Claude credentials the assistant answers from the same facts and the best passage, so the
  dashboard works without a key. To enable Claude, set `ANTHROPIC_API_KEY` (or run `ant auth login`).

In [1]:
import warnings, base64, io, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from fastapi.testclient import TestClient
from urbanheat.app import assistant
from urbanheat.app.server import create_app, PRESETS
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 90
app = create_app()                     # dataset from the config, model runs/m3_lite16_kd
ws, api = app.state.ws, TestClient(app)
PID = "EMIT_243"
lay = api.get(f"/api/layer/{PID}/lst").json()
(s, w), (n, e) = lay["bounds"]
cy, cx, d = (s + n) / 2, (w + e) / 2, (n - s) * 0.12
DISTRICT = {"type": "Polygon", "coordinates": [[[cx - d, cy - d], [cx + d, cy - d], [cx + d, cy + d], [cx - d, cy + d], [cx - d, cy - d]]]}
print(ws.run, "|", len(ws.info), "cities | district of", PID, "around", round(cy, 4), round(cx, 4))

m3_lite16_kd | 422 cities | district of EMIT_243 around 10.9349 -74.8251


## The knowledge base

In [2]:
kb = ws.knowledge
pd.DataFrame([(s, t, len(x)) for s, t, x in kb.passages], columns=["source", "passage", "chars"]).groupby("source").agg(
    passages=("passage", "count"), chars=("chars", "sum"))

,passages,chars
source,,
CONTRACTS.md,7,4299
HANDOVER_MEMBER3.md,11,7456
KNOWLEDGE.md,18,6756


## Retrieval for the questions on slide 22

In [3]:
for q in ["Why is this area predicted to be hot?", "Which factors influence this prediction?",
          "What does a high NDBI value indicate?", "Is this the air temperature people feel?",
          "Would planting trees here cool it, and can I trust that?"]:
    print(q); [print(f"   {h['score']:.2f}  {h['title']}  ({h['source']})") for h in kb.search(q, 3)]

Why is this area predicted to be hot?
   0.09  Thermal context  (KNOWLEDGE.md)
   0.08  5. Map outputs — for the dashboard  (CONTRACTS.md)
   0.06  Mitigation options for surface heat  (KNOWLEDGE.md)
Which factors influence this prediction?
   0.18  What-if scenarios  (KNOWLEDGE.md)
   0.14  What drives urban heat in this data  (KNOWLEDGE.md)
   0.12  Explanations: integrated gradients and Grad-CAM  (KNOWLEDGE.md)
What does a high NDBI value indicate?
   0.25  NDBI: built-up surfaces  (KNOWLEDGE.md)
   0.08  LST anomaly: what the model predicts  (KNOWLEDGE.md)
   0.05  MNDWI: water  (KNOWLEDGE.md)
Is this the air temperature people feel?
   0.29  Land surface temperature (LST)  (KNOWLEDGE.md)
   0.17  5. Limits (please show these in the dashboard and the report)  (HANDOVER_MEMBER3.md)
   0.17  Elevation and slope  (KNOWLEDGE.md)
Would planting trees here cool it, and can I trust that?
   0.21  Mitigation options for surface heat  (KNOWLEDGE.md)
   0.09  MNDWI: water  (KNOWLEDGE.md)
   

## Facts for a district, and the prompt Claude receives

In [4]:
facts = ws.explain(PID, DISTRICT)["facts"]
hits = kb.search(f"Why is this area predicted to be hot? {facts['top_factors']}", 4)
print(assistant.SYSTEM, "\n")
print("FACTS\n" + assistant.format_facts(facts))
print("\nPASSAGES\n" + "\n".join(f"[{i}] {h['title']}: {h['text'][:160]}..." for i, h in enumerate(hits, 1)))

You explain urban heat maps to city planners and students. Answer from the FACTS (model outputs for the place the user selected) and the numbered PASSAGES only. Cite passages as [1], [2]. Give numbers with units (deg C). Say plainly that predictions are land surface temperature, not air temperature, and that attributions and scenarios are model estimates, not proof of cause. If the facts and passages do not answer the question, say so. Keep it under 200 words, in plain language. 

FACTS
- city: EMIT_243
- sensor: EMIT
- climate: tropical
- landsat_date: 2025-03-11
- region: drawn polygon
- pixels: 1023
- city_median_lst_c: 45.23
- predicted_anomaly_c: +0.77 °C
- observed_anomaly_c: +1.11 °C
- top_factors: thermal context +0.88 °C, landsat bands -0.06 °C, hyperspectral -0.05 °C, built_share +0.04 °C

PASSAGES
[1] Thermal context: The thermal context input is the median LST of other Landsat dates in the same season (±60 days of the year), leaving out ±16 days around the target date, so t

## Answers

In [5]:
for q in ["Why is this area predicted to be hot?", "What does a high NDBI value indicate?"]:
    r = api.post("/api/ask", json={"question": q, "pid": PID, "geometry": DISTRICT}).json()
    print(f"Q: {q}\n[{r['mode']}] {r['answer']}\nsources: {[s['title'] for s in r['sources']]}\n")

Q: Why is this area predicted to be hot?
[offline] The model's main factors here: thermal context +0.88 °C, landsat bands -0.06 °C, hyperspectral -0.05 °C, built_share +0.04 °C. Predicted LST anomaly: +0.77 °C (relative to the city median). Background [1] Thermal context: The thermal context input is the median LST of other Landsat dates in the same season (±60 days of the year), leaving out ±16 days around the target date, so the target is never seen. It tells the model where a city is usually hot. It is the strongest single input (alone it explains about 73 % of the anomaly variance with a linear model) but it explains little about why a place is hot: it carries  These are modelled land surface temperatures, not air temperatures or proof of cause.
sources: ['Thermal context', 'Hyperspectral data', 'The project models', 'Limits of the scenario results']

Q: What does a high NDBI value indicate?
[offline] The model's main factors here: thermal context +0.88 °C, landsat bands -0.06 °C, 

This notebook ran without Claude credentials, so the answers above are the **offline** mode. With
`ANTHROPIC_API_KEY` set, the same call returns Claude's wording of the same facts and passages
(`mode: "claude"`). The API path is covered by `tests/test_app.py::test_ask_uses_claude_when_available`.